In [24]:
import geopandas as gpd
import pandas as pd 
import numpy as np
import rasterio

from rasterio.features import rasterize
from rasterio.transform import from_origin


In [25]:
gdf = gpd.read_file('../data/input_datasets/Deforestation_Drivers_Data_archive_from_Camilo/PER_cacao_point.shp')

In [26]:

def rasterize_points(gdf, out_path, pixel_size=1.0, value=1):
    """
    Rasterize a set of points given as a GeoDataFrame.

    Parameters:
    gdf (GeoDataFrame): GeoDataFrame containing the points to rasterize.
    out_path (str): Path to save the output raster file.
    pixel_size (float): The pixel size of the output raster.
    value (int or float): The value to assign to the rasterized points.

    Returns:
    None
    """
    # Get the bounds of the GeoDataFrame
    bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
    
    # Define the raster size and transform
    width = int((bounds[2] - bounds[0]) / pixel_size)
    height = int((bounds[3] - bounds[1]) / pixel_size)
    transform = from_origin(bounds[0], bounds[3], pixel_size, pixel_size)
    
    # Create an empty raster
    raster = rasterio.open(
        out_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='int32',
        compress='lzw',  # or 'deflate', 'zstd'

        crs=gdf.crs.to_string(),
        transform=transform
    )
    
    # Generate (geometry, value) pairs
    shapes = ((geom, value) for geom in gdf.geometry)
    
    # Rasterize the points
    rasterized = rasterize(shapes=shapes, out_shape=(height, width), transform=transform, fill=0)
    
    # Write the raster to the file
    raster.write(rasterized, 1)
    raster.close()

In [27]:
gdf = gdf.to_crs(gdf.estimate_utm_crs())

In [28]:
rasterize_points(gdf, '../data/tmp/cocoa_peru_200.tif', pixel_size=200, value=1)

In [2]:
with rasterio.open('../data/tmp/cocoa_peru_lzw.tif') as src:
    data = src.read()
    profile = src.profile

In [3]:
data.shape

(1, 15437, 11247)

In [4]:
profile

{'driver': 'GTiff', 'dtype': 'int32', 'nodata': None, 'width': 11247, 'height': 15437, 'count': 1, 'crs': CRS.from_epsg(32718), 'transform': Affine(100.0, 0.0, 36965.673532129265,
       0.0, -100.0, 9973025.54577757), 'blockysize': 1, 'tiled': False, 'compress': 'lzw', 'interleave': 'band'}

In [20]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

def reproject_raster(input_raster_path, output_raster_path,dst_crs='EPSG:4326'):
    """
    Reprojects a raster 

    Parameters:
    input_raster_path (str): Path to the input raster file.
    output_raster_path (str): Path to the output raster file.
    """
    # Open the input raster
    with rasterio.open(input_raster_path) as src:
        # Get the original CRS
        src_crs = src.crs

        
        # Calculate the transform and new dimensions
        transform, width, height = calculate_default_transform(
            src_crs, dst_crs, src.width, src.height, *src.bounds)
        
        # Define the metadata for the output raster
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height
        })
        
        # Create the output raster and reproject the data
        with rasterio.open(output_raster_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src_crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    compress='lzw',
                    resampling=Resampling.nearest)



In [29]:
reproject_raster('../data/tmp/cocoa_peru_200.tif', '../data/input_datasets/cocoa_peru_200_4326.tif')